# Testes interativos - Camada de Persistencia

Sistema de Agendamento de Barbearia (JPA + Hibernate + H2)

Este notebook usa o kernel **IJava**. Antes de executar, compile o projeto com `mvn compile` para gerar `../target/classes` (classes + `META-INF/persistence.xml`).

Relacoes testadas:
- **1:N** - Barbearia -> Barbeiro
- **1:1** - Cliente -> Endereco
- **N:M** - Agendamento <-> Servico (via classe associativa ItemAgendamento)

In [ ]:
// Dependencias necessarias para JPA/Hibernate + H2
%maven org.hibernate.orm:hibernate-core:6.4.1.Final
%maven jakarta.persistence:jakarta.persistence-api:3.1.0
%maven com.h2database:h2:2.2.224
%maven org.hibernate.validator:hibernate-validator:8.0.1.Final
%maven org.glassfish:jakarta.el:4.0.2
%maven org.slf4j:slf4j-api:2.0.12
%maven ch.qos.logback:logback-classic:1.4.14

In [ ]:
// Adicionando as classes compiladas do projeto (inclui META-INF/persistence.xml)
%classpath add jar ../target/classes

import br.ufg.inf.barbearia.model.Agendamento;
import br.ufg.inf.barbearia.model.Barbearia;
import br.ufg.inf.barbearia.model.Barbeiro;
import br.ufg.inf.barbearia.model.Cliente;
import br.ufg.inf.barbearia.model.Endereco;
import br.ufg.inf.barbearia.model.Servico;
import br.ufg.inf.barbearia.model.StatusAgendamento;
import br.ufg.inf.barbearia.util.BarbeariaJPAUtil;

import jakarta.persistence.EntityManager;

import java.math.BigDecimal;
import java.time.LocalDate;
import java.time.LocalDateTime;

System.out.println("Classes importadas com sucesso!");

## 1. Relacao 1:N - Barbearia possui varios Barbeiros

In [ ]:
EntityManager em = BarbeariaJPAUtil.createEntityManager();
try {
    em.getTransaction().begin();

    Barbearia barbearia = new Barbearia();
    barbearia.setNome("Barbearia Estilo e Navalha");
    barbearia.setTelefone("(62) 3333-4444");
    barbearia.setEndereco("Av. Central, 100 - Goiania/GO");
    em.persist(barbearia);

    Barbeiro carlos = new Barbeiro();
    carlos.setNome("Carlos Souza");
    carlos.setEspecialidade("Cortes classicos e barba");
    carlos.setDataContratacao(LocalDate.of(2022, 3, 1));
    carlos.setBarbearia(barbearia);
    em.persist(carlos);

    Barbeiro ana = new Barbeiro();
    ana.setNome("Ana Lima");
    ana.setEspecialidade("Cortes modernos");
    ana.setDataContratacao(LocalDate.of(2023, 6, 15));
    ana.setBarbearia(barbearia);
    em.persist(ana);

    em.getTransaction().commit();
    System.out.println("Barbearia criada com id " + barbearia.getId() + " e " + barbearia.getBarbeiros().size() + " barbeiro(s)");
} finally {
    BarbeariaJPAUtil.closeEntityManager(em);
}

In [ ]:
// Verificando o relacionamento 1:N a partir do lado "um"
EntityManager em2 = BarbeariaJPAUtil.createEntityManager();
try {
    Barbearia encontrada = em2.createQuery(
        "select b from Barbearia b where b.nome = :nome", Barbearia.class)
        .setParameter("nome", "Barbearia Estilo e Navalha")
        .getSingleResult();

    System.out.println("Barbearia: " + encontrada.getNome());
    for (Barbeiro b : encontrada.getBarbeiros()) {
        System.out.println("  -> Barbeiro: " + b.getNome() + " (" + b.getEspecialidade() + ")");
    }

    assert encontrada.getBarbeiros().size() == 2 : "Esperado 2 barbeiros";
    System.out.println("OK: relacao 1:N validada");
} finally {
    BarbeariaJPAUtil.closeEntityManager(em2);
}

## 2. Relacao 1:1 - Cliente possui um unico Endereco

In [ ]:
EntityManager em3 = BarbeariaJPAUtil.createEntityManager();
Long clienteId;
try {
    em3.getTransaction().begin();

    Cliente cliente = new Cliente();
    cliente.setNome("Joao Pereira");
    cliente.setTelefone("(62) 99999-1111");
    cliente.setEmail("joao.pereira@example.com");
    em3.persist(cliente);

    Endereco endereco = new Endereco();
    endereco.setRua("Rua das Flores");
    endereco.setNumero("123");
    endereco.setBairro("Setor Central");
    endereco.setCidade("Goiania");
    endereco.setCep("74000-000");
    endereco.setCliente(cliente);
    em3.persist(endereco);

    em3.getTransaction().commit();
    clienteId = cliente.getId();
    System.out.println("Cliente " + cliente.getNome() + " criado com endereco em " + endereco.getCidade());
} finally {
    BarbeariaJPAUtil.closeEntityManager(em3);
}

In [ ]:
// Consultando o endereco via JPQL a partir do cliente
EntityManager em4 = BarbeariaJPAUtil.createEntityManager();
try {
    Endereco encontrado = em4.createQuery(
        "select e from Endereco e where e.cliente.id = :id", Endereco.class)
        .setParameter("id", clienteId)
        .getSingleResult();

    System.out.println("Cliente: " + encontrado.getCliente().getNome());
    System.out.println("Endereco: " + encontrado.getRua() + ", " + encontrado.getNumero() + " - " + encontrado.getCidade());

    assert encontrado.getCliente().getId().equals(clienteId);
    System.out.println("OK: relacao 1:1 validada");
} finally {
    BarbeariaJPAUtil.closeEntityManager(em4);
}

## 3. Relacao N:M - Agendamento inclui varios Servicos (e um Servico aparece em varios Agendamentos)

Implementada via classe associativa `ItemAgendamento`, que tambem guarda o `precoCobrado` no momento do agendamento.

In [ ]:
EntityManager em5 = BarbeariaJPAUtil.createEntityManager();
Long agendamentoId;
try {
    em5.getTransaction().begin();

    Barbearia barbearia = em5.createQuery("select b from Barbearia b", Barbearia.class).getResultList().get(0);
    Barbeiro barbeiro = barbearia.getBarbeiros().get(0);
    Cliente cliente = em5.createQuery("select c from Cliente c", Cliente.class).getResultList().get(0);

    Servico corte = new Servico();
    corte.setNome("Corte de cabelo");
    corte.setDuracaoMinutos(30);
    corte.setPreco(new BigDecimal("40.00"));
    em5.persist(corte);

    Servico barba = new Servico();
    barba.setNome("Barba");
    barba.setDuracaoMinutos(20);
    barba.setPreco(new BigDecimal("25.00"));
    em5.persist(barba);

    Agendamento agendamento = new Agendamento();
    agendamento.setDataHora(LocalDateTime.now().plusDays(1));
    agendamento.setStatus(StatusAgendamento.AGENDADO);
    agendamento.setCliente(cliente);
    agendamento.setBarbeiro(barbeiro);
    agendamento.adicionarServico(corte);
    agendamento.adicionarServico(barba);
    em5.persist(agendamento);

    // Um segundo agendamento reaproveitando o servico "Corte de cabelo" (mostra o lado N do Servico)
    Agendamento segundoAgendamento = new Agendamento();
    segundoAgendamento.setDataHora(LocalDateTime.now().plusDays(2));
    segundoAgendamento.setStatus(StatusAgendamento.AGENDADO);
    segundoAgendamento.setCliente(cliente);
    segundoAgendamento.setBarbeiro(barbeiro);
    segundoAgendamento.adicionarServico(corte);
    em5.persist(segundoAgendamento);

    em5.getTransaction().commit();
    agendamentoId = agendamento.getId();
    System.out.println("Agendamento " + agendamento.getId() + " criado com " + agendamento.getItens().size() + " servico(s)");
} finally {
    BarbeariaJPAUtil.closeEntityManager(em5);
}

In [ ]:
// Validando o N:M nos dois sentidos
EntityManager em6 = BarbeariaJPAUtil.createEntityManager();
try {
    Agendamento agendamento = em6.find(Agendamento.class, agendamentoId);
    System.out.println("Agendamento em " + agendamento.getDataHora() + " inclui:");
    agendamento.getItens().forEach(item ->
        System.out.println("  -> " + item.getServico().getNome() + " (R$ " + item.getPrecoCobrado() + ")"));
    assert agendamento.getItens().size() == 2 : "Esperado 2 servicos no agendamento";

    Servico corte = em6.createQuery(
        "select s from Servico s where s.nome = :nome", Servico.class)
        .setParameter("nome", "Corte de cabelo")
        .getSingleResult();
    System.out.println();
    System.out.println("Servico '" + corte.getNome() + "' aparece em " + corte.getItens().size() + " agendamento(s)");
    assert corte.getItens().size() == 2 : "Esperado que o corte aparecesse em 2 agendamentos";

    System.out.println();
    System.out.println("OK: relacao N:M validada nos dois sentidos");
} finally {
    BarbeariaJPAUtil.closeEntityManager(em6);
}

## 4. Encerrando os recursos

In [ ]:
BarbeariaJPAUtil.closeEntityManagerFactory();
System.out.println("EntityManagerFactory encerrado.");